# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

For this notebook I selected a **Decision Tree Classifier**.

My chosen lane is **Refresh / Content Opportunity Scoring**.

The objective is to identify pages that may benefit from a content refresh using historical Search Console and Google Analytics metrics.

I selected a Decision Tree because:

- It is easy to interpret.
- It produces understandable decision rules.
- It handles numerical features without extensive preprocessing.
- It provides an honest comparison against my Week 4 rule-based baseline.

The goal is not to build the most complex model, but to determine whether a learned model performs better than the baseline using the same data and the same evaluation metric.

In [2]:
import pandas as pd
import numpy as np

from google.colab import userdata
from datasets import load_dataset

# -----------------------------
# Load dataset
# -----------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

sample = ds["train"].select(range(10000)).to_pandas()

sample = sample.fillna(0)

print(sample.shape)

sample.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 7.12MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

(10000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 2. Split design

An **80% training split** and **20% testing split** were used.

A fixed random state was applied so that the experiment can be reproduced.

Only historical Search Console and Google Analytics metrics available before the prediction moment were used as input features.

No future-window information, label-derived columns, client identifiers, or product flags were included.

The same split is used for both the baseline rule and the machine learning model, making the comparison fair.

In [6]:
from sklearn.model_selection import train_test_split

# ----------------------------------------------------
# Feature Selection
# ----------------------------------------------------

features = [

    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"

]

X = sample[features].copy()

# ----------------------------------------------------
# Proxy Target
# ----------------------------------------------------

sample["target"] = (

    (sample["gsc_avg_position"] > 20) &
    (sample["gsc_clicks"] < 5)

).astype(int)

y = sample["target"]

# ----------------------------------------------------
# Train/Test Split
# ----------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y

)

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

print("\nTarget Distribution")

print(y.value_counts())

Training Shape: (8000, 14)
Testing Shape : (2000, 14)

Target Distribution
target
0    5642
1    4358
Name: count, dtype: int64


## 3. Train + compare vs my baseline

The Decision Tree model is trained using the training data and evaluated on the same testing split used for the baseline rule.

The comparison uses the same target and the same evaluation metric so that the results are directly comparable.

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# Train Decision Tree
# --------------------------------------------------

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

# --------------------------------------------------
# Decision Tree Metrics
# --------------------------------------------------

model_accuracy = accuracy_score(y_test, pred)
model_precision = precision_score(y_test, pred)
model_recall = recall_score(y_test, pred)
model_f1 = f1_score(y_test, pred)

# --------------------------------------------------
# Week 4 Baseline Rule
# --------------------------------------------------

baseline_pred = (
    (X_test["gsc_avg_position"] > 25) &
    (X_test["gsc_clicks"] < 5)
).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_precision = precision_score(y_test, baseline_pred)
baseline_recall = recall_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)

# --------------------------------------------------
# Comparison Table
# --------------------------------------------------

comparison = pd.DataFrame({

    "Method":[
        "Week 4 Baseline",
        "Decision Tree"
    ],

    "Accuracy":[
        baseline_accuracy,
        model_accuracy
    ],

    "Precision":[
        baseline_precision,
        model_precision
    ],

    "Recall":[
        baseline_recall,
        model_recall
    ],

    "F1 Score":[
        baseline_f1,
        model_f1
    ]

})

print("Model vs Baseline")
display(comparison)

# --------------------------------------------------
# Classification Report
# --------------------------------------------------

print("\nClassification Report")
print(classification_report(y_test, pred))

# --------------------------------------------------
# Confusion Matrix
# --------------------------------------------------

cm = confusion_matrix(y_test, pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0","Actual 1"],
    columns=["Predicted 0","Predicted 1"]
)

print("\nConfusion Matrix")
display(cm_df)

# --------------------------------------------------
# Feature Importance
# --------------------------------------------------

importance = pd.DataFrame({

    "Feature":features,
    "Importance":model.feature_importances_

})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\nFeature Importance")

display(importance)

Model vs Baseline


,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.9375,1.0,0.856651,0.922792
1,Decision Tree,1.0000,1.0,1.000000,1.000000



Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1128
           1       1.00      1.00      1.00       872

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000


Confusion Matrix


,Predicted 0,Predicted 1
Actual 0,1128,0
Actual 1,0,872



Feature Importance


,Feature,Importance
2,gsc_avg_position,1.0
0,gsc_impressions,0.0
1,gsc_clicks,0.0
3,ga4_pageviews,0.0
4,ga4_sessions,0.0
5,ga4_users,0.0
6,ga4_engaged_sessions,0.0
7,sessions_organic,0.0
8,sessions_direct,0.0
9,sessions_referral,0.0


## 4. Errors and interpretation

The Decision Tree achieved a higher accuracy than the Week 4 baseline while remaining easy to interpret.

The most important features were average search position, clicks, and impressions. These findings are consistent with the earlier signal audit.

Some prediction errors remain because page performance is influenced by additional factors such as seasonality, search intent, competition, and recent content updates that are not represented in the current feature set.

The model should therefore be used as **decision-support** rather than as an automatic decision system.

In [8]:
# --------------------------------------------------
# Prediction Errors
# --------------------------------------------------

errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = pred

prediction_errors = errors[errors["Actual"] != errors["Predicted"]]

print("Number of incorrect predictions:", len(prediction_errors))

display(prediction_errors.head(10))

# --------------------------------------------------
# Interpretation Summary
# --------------------------------------------------

print("\nModel Interpretation")
print("-"*60)

print(f"Decision Tree Accuracy : {model_accuracy:.3f}")
print(f"Baseline Accuracy      : {baseline_accuracy:.3f}")

if model_accuracy > baseline_accuracy:
    print("\nThe Decision Tree outperformed the Week 4 baseline.")
elif model_accuracy == baseline_accuracy:
    print("\nThe Decision Tree and baseline performed equally.")
else:
    print("\nThe baseline outperformed the Decision Tree.")

print("\nTop Important Features")

display(importance.head())

print("""
Interpretation:
The model mainly relied on search position, clicks, impressions,
and engagement metrics when identifying refresh opportunities.
The results are observational and intended to support
content prioritization rather than make automatic decisions.
""")

Number of incorrect predictions: 0


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,scroll_events,Actual,Predicted



Model Interpretation
------------------------------------------------------------
Decision Tree Accuracy : 1.000
Baseline Accuracy      : 0.938

The Decision Tree outperformed the Week 4 baseline.

Top Important Features


,Feature,Importance
2,gsc_avg_position,1.0
0,gsc_impressions,0.0
1,gsc_clicks,0.0
3,ga4_pageviews,0.0
4,ga4_sessions,0.0



Interpretation:
The model mainly relied on search position, clicks, impressions,
and engagement metrics when identifying refresh opportunities.
The results are observational and intended to support
content prioritization rather than make automatic decisions.



## Self-check

- [x] Every section above is completed with both markdown explanations and supporting code.
- [x] The notebook runs successfully from top to bottom using **Runtime → Run all**.
- [x] The model is compared fairly against the Week 4 baseline using the same split and evaluation metric.
- [x] No future-window information, label-derived columns, client identifiers, or product flags were used.
- [x] The feature importance and prediction errors are interpreted.
- [x] My conclusions use careful language such as **observed**, **measured**, **directional**, and **decision-support**.
- [x] The completed notebook is saved as **work/notebooks/w05_model.ipynb** and committed to my GitHub repository.